# SI 671 — RQ2 Temporal Demand Profile Mining

**Yiwu Gu | 2025 NYC Yellow Taxi**  
The authoritative reproducible algorithms are in `src/temporal/`. Run the three commands below in your **repo-root terminal** before inspecting results in this notebook.

```bash
python -m src.temporal.build_profiles --threads 4
python -m src.temporal.cluster_profiles --variance 0.90 --min-pickups 1
python -m src.temporal.plot_results
```

Study design: 263 valid geographic taxi zones; use Pickup-valid records, not `valid_od`; 168D Monday=0...Sunday=6 mean weekday-hour counts; exclude 2025-03-09 and 2025-11-02 (DST dates); row-normalize to study shape; fit PCA and K-means; select K using silhouette with small-cluster guard.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import Image, display
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
TABLES=ROOT/'results/temporal/tables'
FIGS=ROOT/'results/temporal/figures'
assert TABLES.exists(), 'Run build_profiles and cluster_profiles first.'
meta=json.loads((TABLES/'profile_metadata.json').read_text())
cluster_meta=json.loads((TABLES/'clustering_metadata.json').read_text())
print('Source metadata:',meta)
print('Clustering metadata:',cluster_meta)


## 1. Sanity checks: 263 × 168 profiles

Raw = mean demand per day-of-week and hour; Normalized = within-zone relative shape. The latter is our primary K-means input.

In [ ]:
raw=pd.read_csv(TABLES/'zone_profiles_raw.csv')
norm=pd.read_csv(TABLES/'zone_profiles_normalized.csv')
diag=pd.read_csv(TABLES/'zone_profile_diagnostics.csv')
assert raw.shape==norm.shape==(263,169)
print('Active zones:',int((diag.total_pickups>0).sum()))
print('Normalized row sums:',norm.drop(columns='zone_id').sum(axis=1).describe())
display(diag.sort_values('total_pickups',ascending=False).head(10))


## 2. PCA and choosing K

Silhouette is evaluated **in the same retained PCA space as K-means**. K is an analytical choice, not ground-truth class count.

In [ ]:
ks=pd.read_csv(TABLES/'k_selection.csv')
summary=pd.read_csv(TABLES/'cluster_summary.csv')
print('Selected K:',cluster_meta['chosen_k'],'PCA components:',cluster_meta['pca_n_components'])
display(ks)
display(summary)
for name in ['rq2_silhouette_k.png','rq2_pca_variance.png','rq2_pca_scatter.png']:
    p=FIGS/name
    if p.exists(): display(Image(filename=str(p),width=650))


## 3. Interpret cluster-average temporal profiles

Cluster IDs are arbitrary. Name them only after inspecting the profiles, peak hours, weekly structure and representative locations.

In [ ]:
reps=pd.read_csv(TABLES/'cluster_representatives.csv')
display(reps)
for name in ['rq2_cluster_168d_heatmaps.png','rq2_cluster_time_curves.png','rq2_temporal_cluster_map.png']:
    p=FIGS/name
    if p.exists():display(Image(filename=str(p),width=800))


## 4. Compare RQ1 mobility communities vs RQ2 temporal clusters

RQ1 clusters are based on OD flow connectivity. RQ2 clusters are based on pickup demand shape. ARI/AMI and the contingency table tell whether they align on the **same eligible geographic zones**. They do not imply causality.

In [ ]:
p=TABLES/'community_cluster_crosstab.csv'
if p.exists():
    display(pd.read_csv(p))
    print('Agreement metrics:',cluster_meta.get('rq1_rq2_agreement'))
    picture=FIGS/'rq1_rq2_overlap.png'
    if picture.exists(): display(Image(filename=str(picture),width=620))
else: print('Optional RQ1 comparison unavailable: check results/network/tables/communities_annual.csv')


## 5. Notes for the final report

Record: (i) included Pickup-valid records and DST policy, (ii) 263×168 profile construction, (iii) row normalization and zero-zone handling, (iv) PCA dimensions / cumulative variance, (v) selected K, silhouette and smallest cluster, (vi) cluster peak times, representative zones and NYC map, (vii) seed stability, (viii) descriptive RQ1/RQ2 ARI/AMI, (ix) limitation that Yellow Taxi demand is not full urban mobility.

**Leakage warning:** these cluster labels use full-year 2025 demand. Do *not* feed them into 2025 forecasting as historical features.